In [ ]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
src_dir = cwd / "src" if (cwd / "src").exists() else cwd.parent / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from utils.data_processing import (
    extract_condition_coc, 
    extract_day_coc,
    extract_hours_prior_coc
)

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import numpy as np

COND_ORDER = ["+FGF", "-FGF"]
CONDITION_PALETTE = {"+FGF": "#2E86AB", "-FGF": "#A23B72"}
MIN_MAX_CELL_VOLUME = (500, 6500) #seems safe after exploring the volume distribution


def set_hue_legend(ax_or_grid, title, loc="upper left", bbox_to_anchor=(1.01, 1)):
    """Update an existing hue legend title without redrawing (avoids overlapping titles)."""
    # JointGrid
    if hasattr(ax_or_grid, "ax_joint"):
        ax = ax_or_grid.ax_joint
        if ax.get_legend() is not None:
            sns.move_legend(ax, loc, bbox_to_anchor=bbox_to_anchor, title=title)
        return

    # Matplotlib Axes (e.g. stripplot).
    # Check get_legend() before .legend — Axes.legend is a method, not a Legend object.
    if hasattr(ax_or_grid, "get_legend"):
        legend = ax_or_grid.get_legend()
        if legend is not None:
            sns.move_legend(ax_or_grid, loc, bbox_to_anchor=bbox_to_anchor, title=title)
            return

    # FacetGrid / catplot (legend lives on _legend; .legend may be absent or a property)
    legend = getattr(ax_or_grid, "_legend", None)
    legend_attr = getattr(ax_or_grid, "legend", None)
    if legend is None and legend_attr is not None and not callable(legend_attr):
        legend = legend_attr
    if legend is not None:
        legend.set_title(title)

In [ ]:
# Notebook runs from notebooks/, so results is one level up
results_dir = Path("../coc_results").resolve()
output_dir = results_dir / "processed_csvs"
output_file = output_dir / "processed_coc.csv"

# Find every CSV under results/ recursively, excluding the final output file if it already exists
csv_files = sorted(
    csv_path
    for csv_path in results_dir.rglob("*.csv")
    if csv_path.resolve() != output_file.resolve()
)

if not csv_files:
    raise FileNotFoundError(f"No CSV files found under {results_dir}")

# Read and merge all CSV files
merged_df = pd.concat((pd.read_csv(csv_file) for csv_file in csv_files), ignore_index=True)

# Extract 'condition', 'day' and 'magnification' metadata columns and store them
merged_df["condition"] = merged_df["lif_image_name"].apply(extract_condition_coc)
merged_df["day"] = merged_df["lif_container_id"].apply(extract_day_coc)
merged_df["hours_prior_toHPFa"] = merged_df["lif_container_id"].apply(extract_hours_prior_coc)

# Ensure output directory exists and save merged CSV (unfiltered)
output_dir.mkdir(parents=True, exist_ok=True)
merged_df.to_csv(output_file, index=False)

print(f"Merged {len(csv_files)} CSV files into: {output_file}")
print(f"Merged dataframe shape: {merged_df.shape}")
merged_df.head()

In [ ]:
merged_df.columns

In [ ]:
# Plot: For each population, plot area vs mean intensity
# For A549_green
plot_df_a549 = merged_df[merged_df["population"] == "A549_green"]

g_a549 = sns.JointGrid(
    data=plot_df_a549,
    x="area",
    y="A549_green_mean_int",
)
g_a549.plot_joint(sns.scatterplot, alpha=0.5, s=20, edgecolor=None)
g_a549.plot_marginals(sns.kdeplot, fill=True)
g_a549.set_axis_labels("Area", "A549_green mean intensity")
g_a549.figure.suptitle("A549_green: Area vs Mean Intensity", y=1.02)
sns.despine()
plt.show()

# For HPFa_red
plot_df_hpfa = merged_df[merged_df["population"] == "HPFa_red"]

g_hpfa = sns.JointGrid(
    data=plot_df_hpfa,
    x="area",
    y="HPFa_red_mean_int",
)
g_hpfa.plot_joint(sns.scatterplot, alpha=0.5, s=20, edgecolor=None)
g_hpfa.plot_marginals(sns.kdeplot, fill=True)
g_hpfa.set_axis_labels("Area", "HPFa_red mean intensity")
g_hpfa.figure.suptitle("HPFa_red: Area vs Mean Intensity", y=1.02)
sns.despine()
plt.show()

In [ ]:
g_hpfa = sns.JointGrid(
    data=merged_df,
    x="A549_green_mean_int",
    y="HPFa_red_mean_int",
    hue="population",
)
g_hpfa.plot_joint(sns.scatterplot, alpha=0.5, s=20, edgecolor=None)
g_hpfa.plot_marginals(sns.kdeplot, fill=True, common_norm=False)
g_hpfa.set_axis_labels("A549_green mean intensity", "HPFa_red mean intensity")
g_hpfa.figure.suptitle("A549_green vs HPFa_red mean intensity by population", y=1.02)
set_hue_legend(g_hpfa, "Population")
sns.despine()
plt.show()

In [ ]:
# Filter out rows where area is below 500 or above 6500
filtered_df = merged_df[(merged_df["area"] >= MIN_MAX_CELL_VOLUME[0]) & (merged_df["area"] <= MIN_MAX_CELL_VOLUME[1])]

# Total cell number after filtering (per image)
total_cells_df = (
    filtered_df.groupby(["lif_container_id", "lif_image_name", "population", "day", "condition", "hours_prior_toHPFa"], as_index=False)
    .size()
    .rename(columns={"size": "total_cell_nr"})
)

total_cells_file = output_dir / "total_cell_nr.csv"
total_cells_df.to_csv(total_cells_file, index=False)

# Assign filtered_df to merged_df to ensure all plots use the same source of truth
merged_df = filtered_df

In [ ]:
POP_ORDER = ["A549_green", "HPFa_red"]
DAY_ORDER = sorted(total_cells_df["day"].dropna().unique().astype(int).tolist())
HOURS_ORDER = sorted(total_cells_df["hours_prior_toHPFa"].dropna().unique().astype(int).tolist())

plot_df = total_cells_df.copy()
plot_df["day"] = plot_df["day"].astype(int)
plot_df["hours_prior_toHPFa"] = plot_df["hours_prior_toHPFa"].astype(int)

total_cells_df.head()

In [ ]:
# Total cell number by day and condition (one panel per population)
# Individual image points + mean ± SD
for population in POP_ORDER:
    df_pop = plot_df[plot_df["population"] == population].copy()

    plt.figure(figsize=(8, 5))
    ax = sns.stripplot(
        data=df_pop,
        x="day",
        y="total_cell_nr",
        hue="condition",
        order=DAY_ORDER,
        hue_order=COND_ORDER,
        palette=CONDITION_PALETTE,
        dodge=True,
        jitter=0.25,
        alpha=0.75,
        linewidth=0.7,
        edgecolor="gray",
    )

    summary = (
        df_pop.groupby(["day", "condition"], as_index=False)
        .agg(mean_val=("total_cell_nr", "mean"), sd_val=("total_cell_nr", "std"))
    )

    n_conditions = len(COND_ORDER)
    width = 0.8
    for i, cond in enumerate(COND_ORDER):
        for day in DAY_ORDER:
            sub = summary[(summary["day"] == day) & (summary["condition"] == cond)]
            if sub.empty:
                continue
            base = DAY_ORDER.index(day)
            offset = (i - (n_conditions - 1) / 2) * (width / n_conditions)
            ax.errorbar(
                x=base + offset,
                y=sub["mean_val"].values[0],
                yerr=sub["sd_val"].values[0],
                fmt="o",
                color=CONDITION_PALETTE[cond],
                capsize=6,
                lw=2,
                zorder=10,
                markersize=9,
                mec="black",
                mew=1.2,
            )

    ax.set_title(f"{population}: total cell number by day and condition\n(points=images, large marker=mean±SD)")
    ax.set_xlabel("Day")
    ax.set_ylabel("Total cell number")
    set_hue_legend(ax, "Condition")
    plt.tight_layout()
    plt.show()

In [ ]:
# Total cell number by hours prior to HPFa and condition (one panel per population)
for population in POP_ORDER:
    df_pop = plot_df[plot_df["population"] == population].copy()

    plt.figure(figsize=(8, 5))
    ax = sns.stripplot(
        data=df_pop,
        x="hours_prior_toHPFa",
        y="total_cell_nr",
        hue="condition",
        order=HOURS_ORDER,
        hue_order=COND_ORDER,
        palette=CONDITION_PALETTE,
        dodge=True,
        jitter=0.25,
        alpha=0.75,
        linewidth=0.7,
        edgecolor="gray",
    )

    summary = (
        df_pop.groupby(["hours_prior_toHPFa", "condition"], as_index=False)
        .agg(mean_val=("total_cell_nr", "mean"), sd_val=("total_cell_nr", "std"))
    )

    n_conditions = len(COND_ORDER)
    width = 0.8
    for i, cond in enumerate(COND_ORDER):
        for hours in HOURS_ORDER:
            sub = summary[
                (summary["hours_prior_toHPFa"] == hours) & (summary["condition"] == cond)
            ]
            if sub.empty:
                continue
            base = HOURS_ORDER.index(hours)
            offset = (i - (n_conditions - 1) / 2) * (width / n_conditions)
            ax.errorbar(
                x=base + offset,
                y=sub["mean_val"].values[0],
                yerr=sub["sd_val"].values[0],
                fmt="o",
                color=CONDITION_PALETTE[cond],
                capsize=6,
                lw=2,
                zorder=10,
                markersize=9,
                mec="black",
                mew=1.2,
            )

    ax.set_title(
        f"{population}: total cell number by hours prior to HPFa and condition\n"
        "(points=images, large marker=mean±SD)"
    )
    ax.set_xlabel("Hours prior to HPFa")
    ax.set_ylabel("Total cell number")
    set_hue_legend(ax, "Condition")
    plt.tight_layout()
    plt.show()

In [ ]:
# Total cell number by day and condition, split by hours prior to HPFa (one panel per population)
for population in POP_ORDER:
    df_pop = plot_df[plot_df["population"] == population].copy()

    g = sns.catplot(
        data=df_pop,
        x="day",
        y="total_cell_nr",
        hue="condition",
        col="hours_prior_toHPFa",
        order=DAY_ORDER,
        hue_order=COND_ORDER,
        palette=CONDITION_PALETTE,
        kind="strip",
        dodge=True,
        jitter=0.25,
        alpha=0.75,
        linewidth=0.7,
        edgecolor="gray",
        height=4.5,
        aspect=0.95,
    )

    for ax, hours in zip(g.axes.flat, HOURS_ORDER):
        df_panel = df_pop[df_pop["hours_prior_toHPFa"] == hours]
        summary = (
            df_panel.groupby(["day", "condition"], as_index=False)
            .agg(mean_val=("total_cell_nr", "mean"), sd_val=("total_cell_nr", "std"))
        )
        n_conditions = len(COND_ORDER)
        width = 0.8
        for i, cond in enumerate(COND_ORDER):
            for day in DAY_ORDER:
                sub = summary[(summary["day"] == day) & (summary["condition"] == cond)]
                if sub.empty:
                    continue
                base = DAY_ORDER.index(day)
                offset = (i - (n_conditions - 1) / 2) * (width / n_conditions)
                ax.errorbar(
                    x=base + offset,
                    y=sub["mean_val"].values[0],
                    yerr=sub["sd_val"].values[0],
                    fmt="o",
                    color=CONDITION_PALETTE[cond],
                    capsize=5,
                    lw=2,
                    zorder=10,
                    markersize=8,
                    mec="black",
                    mew=1.2,
                )
        ax.set_xlabel("Day")
        ax.set_ylabel("Total cell number")

    g.figure.suptitle(
        f"{population}: total cell number by day × condition × hours prior to HPFa\n"
        "(points=images, large marker=mean±SD)",
        y=1.05,
    )
    set_hue_legend(g, "Condition")
    plt.show()

In [ ]:
# Total cell number by day × population × hours prior to HPFa
# Separate figures for +FGF, -FGF, and pooled (+FGF & -FGF)
POP_PALETTE = {"A549_green": "#2ca02c", "HPFa_red": "#d62728"}

condition_subsets = [
    ("+FGF", plot_df[plot_df["condition"] == "+FGF"].copy()),
    ("-FGF", plot_df[plot_df["condition"] == "-FGF"].copy()),
    ("+FGF & -FGF (pooled)", plot_df[plot_df["condition"].isin(COND_ORDER)].copy()),
]

for subset_label, df_subset in condition_subsets:
    g = sns.catplot(
        data=df_subset,
        x="day",
        y="total_cell_nr",
        hue="population",
        col="hours_prior_toHPFa",
        order=DAY_ORDER,
        hue_order=POP_ORDER,
        palette=POP_PALETTE,
        kind="strip",
        dodge=True,
        jitter=0.25,
        alpha=0.75,
        linewidth=0.7,
        edgecolor="gray",
        height=4.5,
        aspect=0.95,
    )

    for ax, hours in zip(g.axes.flat, HOURS_ORDER):
        df_panel = df_subset[df_subset["hours_prior_toHPFa"] == hours]
        summary = (
            df_panel.groupby(["day", "population"], as_index=False)
            .agg(mean_val=("total_cell_nr", "mean"), sd_val=("total_cell_nr", "std"))
        )
        n_populations = len(POP_ORDER)
        width = 0.8
        for i, population in enumerate(POP_ORDER):
            for day in DAY_ORDER:
                sub = summary[
                    (summary["day"] == day) & (summary["population"] == population)
                ]
                if sub.empty:
                    continue
                base = DAY_ORDER.index(day)
                offset = (i - (n_populations - 1) / 2) * (width / n_populations)
                ax.errorbar(
                    x=base + offset,
                    y=sub["mean_val"].values[0],
                    yerr=sub["sd_val"].values[0],
                    fmt="o",
                    color=POP_PALETTE[population],
                    capsize=5,
                    lw=2,
                    zorder=10,
                    markersize=8,
                    mec="black",
                    mew=1.2,
                )
        ax.set_xlabel("Day")
        ax.set_ylabel("Total cell number")

    g.figure.suptitle(
        f"{subset_label}: total cell number by day × population × hours prior to HPFa\n"
        "(points=images, large marker=mean±SD)",
        y=1.05,
    )
    set_hue_legend(g, "Population")
    plt.show()

In [ ]:
# Total cell number by population and condition (pooled across day / hours)
plt.figure(figsize=(8, 5))
ax = sns.stripplot(
    data=plot_df,
    x="population",
    y="total_cell_nr",
    hue="condition",
    order=POP_ORDER,
    hue_order=COND_ORDER,
    palette=CONDITION_PALETTE,
    dodge=True,
    jitter=0.25,
    alpha=0.75,
    linewidth=0.7,
    edgecolor="gray",
)

summary = (
    plot_df.groupby(["population", "condition"], as_index=False)
    .agg(mean_val=("total_cell_nr", "mean"), sd_val=("total_cell_nr", "std"))
)

n_conditions = len(COND_ORDER)
width = 0.8
for i, cond in enumerate(COND_ORDER):
    for population in POP_ORDER:
        sub = summary[
            (summary["population"] == population) & (summary["condition"] == cond)
        ]
        if sub.empty:
            continue
        base = POP_ORDER.index(population)
        offset = (i - (n_conditions - 1) / 2) * (width / n_conditions)
        ax.errorbar(
            x=base + offset,
            y=sub["mean_val"].values[0],
            yerr=sub["sd_val"].values[0],
            fmt="o",
            color=CONDITION_PALETTE[cond],
            capsize=6,
            lw=2,
            zorder=10,
            markersize=9,
            mec="black",
            mew=1.2,
        )

ax.set_title("Total cell number by population and condition\n(points=images, large marker=mean±SD)")
ax.set_xlabel("Population")
ax.set_ylabel("Total cell number")
set_hue_legend(ax, "Condition")
plt.tight_layout()
plt.show()